# 02 Preprocessing

This notebook prepares clean, model-ready datasets and saves them to data/processed.

In [ ]:
from pathlib import Path
import pandas as pd

from src.data_loader import load_all_raw_datasets
from src.preprocess import (
    clean_dataframe,
    encode_categoricals,
    identify_effort_column,
    scale_numeric_features,
    save_multiple_processed,
    split_features_target,
    )

root_dir = Path.cwd()
raw_datasets = load_all_raw_datasets(root_dir / "data" / "raw")
processed_output = {}

In [ ]:
for name, df in raw_datasets.items():
    effort_col = identify_effort_column(df)
    cleaned = clean_dataframe(df)
    encoded = encode_categoricals(cleaned)

    if effort_col not in encoded.columns:
        effort_col = encoded.columns[-1]

    x, y = split_features_target(encoded, effort_col)
    x_scaled, _ = scale_numeric_features(x)
    processed_df = x_scaled.copy()
    processed_df[effort_col] = pd.to_numeric(y, errors="coerce")
    processed_df = processed_df.dropna(subset=[effort_col])
    processed_output[name] = processed_df

    print(f"{name}: raw={df.shape}, cleaned={cleaned.shape}, processed={processed_df.shape}")

In [ ]:
saved_paths = save_multiple_processed(processed_output, root_dir / "data" / "processed")
print("Saved files:")
for name, path in saved_paths.items():
    print(f"- {name}: {path}")